Ce notebook reprend une version initialemement écrite par Dennis G. Wilson, lui-même inspiré du billet d'Alexander Van de Kleut, *Variational Autoencoders*.

# Auto-encodeurs variationnels (VAE)

**Travaux pratiques – Deep Learning**

Ce notebook prolonge le BE consacré aux auto-encodeurs. Nous repartons de la même base **MNIST**, du même prétraitement (images vectorisées en dimension 784), et des mêmes conventions PyTorch : classes `Encoder`, `Decoder` et `Autoencoder`, données sous forme de couples `(images, targets)` et entraînement avec `DataLoader`.

L'objectif est ensuite de comprendre ce qu'apporte un **auto-encodeur variationnel** (*Variational Autoencoder*, VAE) : au lieu d'associer à chaque image un unique vecteur latent, l'encodeur prédit une distribution dans l'espace latent. Cette représentation permet notamment de générer de nouvelles images en échantillonnant cet espace.

Les cellules contenant du code sont prévues pour être exécutées dans l'ordre. Le téléchargement de MNIST nécessite un accès à Internet.

## 1. Motivation : de la compression à la génération

Un jeu de données d'images est de grande dimension : une image MNIST comporte $28\times28=784$ pixels. Pourtant, les variations pertinentes entre les chiffres peuvent être décrites par un nombre bien plus faible de facteurs : forme, inclinaison, épaisseur du trait, etc.

Un auto-encodeur apprend deux transformations :
- un **encodeur** $e_\phi : \mathcal X\rightarrow\mathcal Z$, qui transforme une observation $x$ en représentation latente $z=e_\phi(x)$ ;
- un **décodeur** $d_\theta : \mathcal Z\rightarrow\mathcal X$, qui reconstruit $\hat x=d_\theta(z)$.

Le réseau complet est entraîné à reconstruire son entrée. Cette approche apprend une représentation compacte, mais elle ne garantit pas que l'espace latent soit régulier : deux codes proches peuvent décoder en images très différentes, et une région arbitraire de l'espace latent peut ne correspondre à aucune image réaliste.

Les VAE introduisent une structure probabiliste qui rend l'espace latent plus régulier et exploitable pour la génération.

## 2. Rappel : auto-encodeur déterministe

Nous reprenons ici l'architecture dense du BE précédent. Les images sont vectorisées, l'encodeur et le décodeur sont définis séparément, puis réunis dans la classe `Autoencoder`.

Dans le BE, la première version utilisait une erreur quadratique moyenne, puis le travail demandé proposait de passer à une formulation de classification binaire. Comme les pixels sont compris entre 0 et 1, nous retenons ici cette seconde formulation : **sigmoïde en sortie et entropie croisée binaire** (`BCELoss`).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilisé :", device)
torch.manual_seed(0)

In [ ]:
from torchvision.datasets import MNIST
from torchvision import transforms

# Chargement de MNIST et normalisation des niveaux de gris entre 0 et 1
transform = transforms.ToTensor()
train_dataset = MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = MNIST(root="./data", train=False, download=True, transform=transform)

# Les images sont vectorisées en dimension 784, comme dans le BE précédent
x_train = train_dataset.data.float().div(255.).view(-1, 784)
x_test = test_dataset.data.float().div(255.).view(-1, 784)

train_loader = DataLoader(TensorDataset(x_train, x_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, x_test), batch_size=128, shuffle=False)

print("Taille du jeu d'entraînement :", x_train.shape)
print("Taille du jeu de test :", x_test.shape)

In [ ]:
# Visualisation de quelques images
n = 10
plt.figure(figsize=(15, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.show()

### 2.1 Architecture de l'auto-encodeur

L'encodeur réduit successivement la dimension $784\rightarrow128\rightarrow d_z$. Le décodeur réalise l'opération inverse. La sigmoïde garantit que les pixels reconstruits appartiennent à l'intervalle $[0,1]$.

La dimension $d_z$ est un hyperparamètre : elle fixe la taille du goulot d'étranglement.

In [ ]:

# Encodeur convolutif
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

    def forward(self, x):
        # Passage de (batch, 784) à (batch, 1, 28, 28)
        x = x.view(-1, 1, 28, 28)
        x = self.conv(x)

        # Vectorisation des caractéristiques convolutives
        x = x.view(x.size(0), -1)
        return self.fc(x)


# Décodeur convolutif
class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64 * 7 * 7),
            nn.ReLU()
        )

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, z):
        x = self.fc(z)
        # Retour à une représentation spatiale
        x = x.view(-1, 64, 7, 7)

        x = self.deconv(x)
        # Retour à un vecteur de dimension 784
        return x.view(-1, 784)


# Auto-encodeur convolutif
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


### 2.2 Entraînement et reconstruction

La fonction de coût mesure l'écart entre l'image d'entrée et sa reconstruction. Nous utilisons ici la somme de l'entropie croisée binaire sur les pixels, puis nous suivons la moyenne par image pour faciliter la comparaison entre jeux de données.

In [ ]:
latent_dim = 2
autoencoder = Autoencoder(latent_dim).to(device)

criterion = nn.BCELoss(reduction="sum")
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

epochs = 10
for epoch in range(epochs):
    autoencoder.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        reconstructions = autoencoder(images)
        loss = criterion(reconstructions, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(f"Époque {epoch+1:02d}/{epochs} - perte BCE/image : "
          f"{train_loss / len(train_loader.dataset):.4f}")

In [ ]:
# Comparaison entre images originales (ligne du haut) et reconstructions (ligne du bas)
autoencoder.eval()
with torch.no_grad():
    decoded_imgs = autoencoder(x_test[:10].to(device)).cpu().numpy()

plt.figure(figsize=(15, 4))
for i in range(10):
    ax = plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    ax.axis("off")
    ax = plt.subplot(2, 10, i + 11)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.show()

Les reconstructions ne sont pas très bonnes, mais nous avons imposé un espace latent volontairement petit (2 dimensions) pour pouvoir effectuer des visualisations dans la partie suivante !

### 2.3 Visualiser et explorer l'espace latent

Avec une dimension latente égale à 2, nous pouvons représenter directement les codes \(z\). Nous colorons les points selon la classe MNIST. Attention : les étiquettes ne sont pas utilisées pour entraîner l'auto-encodeur ; elles servent uniquement à la visualisation.

In [ ]:
# Les étiquettes sont récupérées depuis le jeu de données pour la visualisation
y_test = test_dataset.targets

autoencoder.eval()
with torch.no_grad():
    z_test = autoencoder.encoder(x_test.to(device)).cpu().numpy()

plt.figure(figsize=(8, 6))
plt.scatter(z_test[:, 0], z_test[:, 1], c=y_test.numpy(),
            cmap="tab10", s=5, alpha=0.6)
plt.colorbar()
plt.xlabel("Dimension latente 1")
plt.ylabel("Dimension latente 2")
plt.title("Espace latent de l'auto-encodeur")
plt.show()

Nous pouvons aussi décoder une grille régulière de points de l'espace latent. Cette visualisation permet d'observer ce que le décodeur a appris à associer aux différentes régions de l'espace. Pour un auto-encodeur déterministe, rien ne garantit toutefois que toutes ces régions soient pertinentes.

In [ ]:
autoencoder.eval()

grid_size = 15
grid_x = np.linspace(z_test[:, 0].min(), z_test[:, 0].max(), grid_size)
grid_y = np.linspace(z_test[:, 1].min(), z_test[:, 1].max(), grid_size)

step_x = grid_x[1] - grid_x[0]
step_y = grid_y[1] - grid_y[0]

canvas = np.zeros((28 * grid_size, 28 * grid_size))

with torch.no_grad():
    for i, gy in enumerate(grid_y[::-1]):  # grandes valeurs en haut
        for j, gx in enumerate(grid_x):
            z = torch.tensor([[gx, gy]], dtype=torch.float32, device=device)
            image = autoencoder.decoder(z).cpu().numpy().reshape(28, 28)

            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = image

plt.figure(figsize=(10, 10))

plt.imshow(
    canvas,
    cmap="gray",
    origin="upper",  # conserve l'orientation des chiffres
    extent=[
        grid_x[0] - step_x/2,
        grid_x[-1] + step_x/2,
        grid_y[0] - step_y/2,
        grid_y[-1] + step_y/2
    ],
    aspect="equal"
)

plt.xlabel("Dimension latente 1", fontsize=12)
plt.ylabel("Dimension latente 2", fontsize=12)
plt.title("Décodage d'une grille de points latents")
plt.xticks(np.linspace(grid_x[0], grid_x[-1], 7))
plt.yticks(np.linspace(grid_y[0], grid_y[-1], 7))
plt.show()

## 3. Pourquoi un auto-encodeur variationnel ?

Dans l'auto-encodeur déterministe, l'encodeur associe une seule valeur $z$ à chaque entrée $x$. Le VAE remplace cette valeur par une **distribution conditionnelle** $q_\phi(z\mid x)$.

Dans la version la plus classique, cette distribution est une gaussienne diagonale :

$$
q_\phi(z\mid x)=\mathcal N\left(\mu_\phi(x),
\operatorname{diag}(\sigma_\phi^2(x))\right).
$$

L'encodeur prédit donc deux vecteurs de taille \(d_z\) :
- $\mu_\phi(x)$, le vecteur des moyennes ;
- $\log \sigma_\phi^2(x)$, le vecteur des logarithmes des variances.

Pourquoi prédire le logarithme de la variance ? Cela permet de représenter une variance strictement positive sans imposer directement une contrainte de positivité à la sortie du réseau. On retrouve ensuite $\sigma^2=\exp(\log\sigma^2)$.

### 3.1 Échantillonnage et astuce de reparamétrisation

On souhaite tirer un code latent selon $q_\phi(z\mid x)$. Un échantillonnage direct rendrait la dérivation par rapport aux paramètres de l'encodeur problématique.

L'astuce de reparamétrisation consiste à écrire :

$$
\epsilon\sim\mathcal N(0,I),\qquad
z=\mu+\sigma\odot\epsilon,
$$

où $\sigma=\exp(\frac12\log\sigma^2)$. Le bruit $\epsilon$ est indépendant des paramètres du réseau : le calcul devient différentiable par rapport à $\mu$ et $\log\sigma^2$.

Pendant l'entraînement, on échantillonne donc un code $z$. À l'inférence, on peut également utiliser la moyenne $\mu$ pour obtenir une représentation déterministe.

### 3.2 Régulariser l'espace latent : divergence KL

Il faut éviter que les distributions latentes prédites par l'encodeur soient dispersées de manière arbitraire. On les rapproche d'une loi a priori, généralement la gaussienne standard :

$$
p(z)=\mathcal N(0,I).
$$

On mesure cet écart par la divergence de Kullback-Leibler. Pour une gaussienne diagonale, elle s'écrit :

$$
D_{KL}\big(q_\phi(z\mid x)\,\|\,\mathcal N(0,I)\big)
=-\frac12\sum_{j=1}^{d_z}
\left(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2\right).
$$

Cette pénalisation encourage les codes à occuper une région continue et structurée autour de l'origine. Elle rend possible l'échantillonnage de nouveaux $z\sim\mathcal N(0,I)$, puis leur décodage en nouvelles images.

### 3.3 Fonction objectif : ELBO

L'apprentissage combine deux termes :

1. **Perte de reconstruction** : l'image décodée doit être proche de l'image d'entrée.
2. **Régularisation KL** : la distribution latente doit rester proche de la loi a priori.

On minimise la somme :

$$
\mathcal L_{\mathrm{VAE}}
=\mathcal L_{\mathrm{reconstruction}}
+D_{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big).
$$

Cette quantité correspond à l'opposé de la borne inférieure de vraisemblance (*Evidence Lower Bound*, ELBO), à une convention de signe près. Le compromis est important : une régularisation trop forte peut dégrader les reconstructions ; une régularisation trop faible peut rendre l'espace latent difficile à échantillonner.

## 4. Implémentation du VAE avec PyTorch

Nous conservons les mêmes dimensions d'entrée, les mêmes données MNIST et la même organisation en modules que dans le BE. La différence essentielle est que l'encodeur ne renvoie plus un seul vecteur latent : il renvoie `mu` et `logvar`. Le décodeur reçoit ensuite un échantillon \(z\).

In [ ]:
class EncoderVAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        # A COMPLETER
        # Deux têtes de sortie : moyenne et log-variance

    def forward(self, x):
        # A COMPLETER
        return mu, logvar


class DecoderVAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()

        # A COMPLETER

    def forward(self, z):
        # A COMPLETER


class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        # A COMPLETER

    def reparameterize(self, mu, logvar):
        # Echantillonner la loi de paramètres mus et logvar
        # A COMPLETER
        return

    def forward(self, x):
        # A COMPLETER

### 4.1 Vérifier les dimensions

Pour un mini-lot de taille $B$ et une dimension latente $d_z$, `mu`, `logvar` et `z` sont de taille $B\times d_z$. La reconstruction a la même taille que l'entrée, soit $B\times784$.

In [ ]:
vae = VAE(latent_dim=2).to(device)
x = x_train[:16].to(device)
reconstruction, mu, logvar = vae(x)

print("Entrée             :", x.shape)
print("Moyenne mu         :", mu.shape)
print("Log-variance       :", logvar.shape)
print("Reconstruction     :", reconstruction.shape)

### 4.2 Calcul de la fonction de coût

Nous reprenons l'entropie croisée binaire utilisée pour l'auto-encodeur. Le terme KL est calculé à partir de `mu` et `logvar`, puis sommé sur les dimensions latentes et moyenné sur le mini-lot.

Il est utile de suivre séparément les deux composantes de la perte : une perte totale seule ne permet pas de comprendre le compromis appris par le modèle.

In [ ]:
def vae_loss(reconstruction, x, mu, logvar):
    # Erreur de reconstruction : somme sur les pixels, puis moyenne sur le batch
    reconstruction_loss = ...# A COMPLETER

    # KL(q(z|x) || N(0, I)) : somme sur les dimensions latentes, puis moyenne batch
    kl_loss = ... # A COMPLETER

    return reconstruction_loss + kl_loss, reconstruction_loss, kl_loss

### 4.3 Entraînement

La boucle d'entraînement est très proche de celle du BE. La différence est que le modèle renvoie trois objets : la reconstruction, la moyenne et le logarithme de la variance. Le terme KL s'ajoute à la perte de reconstruction.

In [ ]:
latent_dim = 2
vae = VAE(latent_dim).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

epochs = 15
history = {"total": [], "reconstruction": [], "kl": []}

for epoch in range(epochs):
    vae.train()
    total_epoch = recon_epoch = kl_epoch = 0.0

    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        reconstruction, mu, logvar = vae(images)
        loss, reconstruction_loss, kl_loss = vae_loss(
            reconstruction, targets, mu, logvar
        )
        loss.backward()
        optimizer.step()

        total_epoch += loss.item() * images.size(0)
        recon_epoch += reconstruction_loss.item() * images.size(0)
        kl_epoch += kl_loss.item() * images.size(0)

    n = len(train_loader.dataset)
    history["total"].append(total_epoch / n)
    history["reconstruction"].append(recon_epoch / n)
    history["kl"].append(kl_epoch / n)

    print(f"Époque {epoch+1:02d}/{epochs} | "
          f"totale : {history['total'][-1]:.3f} | "
          f"reconstruction : {history['reconstruction'][-1]:.3f} | "
          f"KL : {history['kl'][-1]:.3f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["total"], label="Perte totale")
plt.plot(history["reconstruction"], label="Reconstruction")
plt.plot(history["kl"], label="Divergence KL")
plt.xlabel("Époque")
plt.ylabel("Perte moyenne par image")
plt.legend()
plt.grid()
plt.show()

### 4.4 Reconstruire des images

Pour reconstruire une image, le VAE encode l'entrée en une distribution puis échantillonne un code latent. Les reconstructions peuvent donc varier légèrement d'un passage à l'autre. Pour supprimer cette variabilité, on peut décoder directement la moyenne $\mu$.

In [ ]:
vae.eval()
with torch.no_grad():
    mu_test, logvar_test = vae.encoder(x_test[:10].to(device))
    z_test_sample = vae.reparameterize(mu_test, logvar_test)
    recon_sample = vae.decoder(z_test_sample).cpu().numpy()
    recon_mean = vae.decoder(mu_test).cpu().numpy()

plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(3, 10, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    plt.subplot(3, 10, i + 11)
    plt.imshow(recon_sample[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    plt.subplot(3, 10, i + 21)
    plt.imshow(recon_mean[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
plt.suptitle("Originales / reconstructions échantillonnées / décodage de la moyenne")
plt.show()

### 4.5 Visualiser l'espace latent du VAE

Pour visualiser les données, nous représentons ici les moyennes $\mu_\phi(x)$ prédites par l'encodeur. Contrairement à l'auto-encodeur déterministe, chaque point est associé à une distribution, et non à un code unique. La régularisation KL tend à organiser ces représentations autour d'une gaussienne standard.

In [ ]:
vae.eval()
with torch.no_grad():
    mu_test, logvar_test = vae.encoder(x_test.to(device))
    mu_test = mu_test.cpu().numpy()

plt.figure(figsize=(8, 6))
plt.scatter(mu_test[:, 0], mu_test[:, 1], c=y_test.numpy(),
            cmap="tab10", s=5, alpha=0.6)
plt.colorbar()
plt.xlabel("Moyenne latente 1")
plt.ylabel("Moyenne latente 2")
plt.title("Moyennes des distributions latentes du VAE")
plt.show()

### 4.6 Générer de nouvelles images

C'est ici que le VAE se distingue nettement de l'auto-encodeur déterministe. Puisque l'espace latent est régularisé pour être proche de $\mathcal N(0,I)$, nous pouvons tirer de nouveaux codes selon cette loi a priori, puis les passer au décodeur.

Les images générées ne sont pas des reconstructions d'images du jeu d'entraînement : elles correspondent à des échantillons nouveaux dans l'espace latent.

In [ ]:
vae.eval()
n_samples = 64
with torch.no_grad():
    z = torch.randn(n_samples, latent_dim, device=device)
    generated = vae.decoder(z).cpu().reshape(-1, 1, 28, 28)

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("Nouvelles images générées par échantillonnage latent")
plt.tight_layout()
plt.show()

### 4.7 Explorer une grille régulière dans l'espace latent

On peut également décoder une grille de coordonnées. Avec un VAE entraîné en dimension latente 2, on explore souvent une région autour de $[-3,3]$ sur chaque axe, correspondant à plusieurs écarts-types de la loi normale standard.

Les transitions progressives entre les images indiquent que le décodeur a appris une représentation relativement continue.

In [ ]:
vae.eval()
grid_size = 15
values = np.linspace(-3, 3, grid_size)

fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
with torch.no_grad():
    for i, gy in enumerate(values[::-1]):
        for j, gx in enumerate(values):
            z = torch.tensor([[gx, gy]], dtype=torch.float32, device=device)
            image = vae.decoder(z).cpu().numpy().reshape(28, 28)
            axes[i, j].imshow(image, cmap="gray")
            axes[i, j].axis("off")
plt.suptitle("Grille d'images générées par le VAE")
plt.tight_layout()
plt.show()

In [ ]:
vae.eval()

grid_size = 15
values = np.linspace(-3, 3, grid_size)
step = values[1] - values[0]

canvas = np.zeros((28 * grid_size, 28 * grid_size))

with torch.no_grad():
    for i, gy in enumerate(values[::-1]):  # grandes valeurs en haut
        for j, gx in enumerate(values):
            z = torch.tensor([[gx, gy]], dtype=torch.float32, device=device)
            image = vae.decoder(z).cpu().numpy().reshape(28, 28)

            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = image

plt.figure(figsize=(10, 10))

plt.imshow(
    canvas,
    cmap="gray",
    origin="upper",  # conserve l'orientation des chiffres
    extent=[
        values[0] - step/2,
        values[-1] + step/2,
        values[0] - step/2,
        values[-1] + step/2
    ],
    aspect="equal"
)

plt.xlabel("Dimension latente 1", fontsize=12)
plt.ylabel("Dimension latente 2", fontsize=12)
plt.title("Génération d'images dans l'espace latent du VAE")
plt.xticks(np.linspace(-3, 3, 7))
plt.yticks(np.linspace(-3, 3, 7))
plt.show()

## 6. À retenir

- Un auto-encodeur déterministe apprend un code latent $z$ et une reconstruction.
- Un VAE apprend une distribution $q_\phi(z\mid x)$, paramétrée par une moyenne et une variance.
- L'astuce de reparamétrisation permet de rétropropager à travers l'échantillonnage.
- La fonction objectif combine reconstruction et divergence KL vers une loi a priori, généralement $\mathcal N(0,I)$.
- Cette régularisation permet de générer de nouvelles observations en échantillonnant l'espace latent.

